In [3]:
import os
import torch
import timm
from PIL import Image, ImageDraw
from torchvision import transforms

# ========= PATH =========
PROJECT_ROOT = r"E:\semester7\comvis\deteksidaun"
MODEL_PATH   = os.path.join(PROJECT_ROOT, "models/best_model.pth")
CLASSES_PATH = os.path.join(PROJECT_ROOT, "models/classes.pt")
OUT_DIR      = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ========= LOAD CLASSES =========
classes = torch.load(CLASSES_PATH, weights_only=True)
num_classes = len(classes)
print("Classes:", classes)

# ========= TRANSFORM =========
mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
infer_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

# ========= LOAD MODEL =========
model = timm.create_model("convnext_small", pretrained=False, num_classes=num_classes)
state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.to(device).eval()

def predict_image(img_path, topk=3, threshold=0.80):
    img = Image.open(img_path).convert("RGB")
    x = infer_tf(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]

    top_probs, top_idx = torch.topk(probs, k=min(topk, num_classes))
    results = []
    for p, i in zip(top_probs, top_idx):
        results.append({"class": classes[i.item()], "confidence": float(p.item())})

    results[0]["status"] = "ok" if results[0]["confidence"] >= threshold else "uncertain"
    return results

def annotate_center_crop(img: Image.Image, label: str, conf: float, crop_size=224):
    w, h = img.size
    left = (w - crop_size) // 2
    top  = (h - crop_size) // 2
    right = left + crop_size
    bottom = top + crop_size

    draw = ImageDraw.Draw(img)
    draw.rectangle([left, top, right, bottom], width=4)

    text = f"{label} ({conf*100:.1f}%)"
    pad = 6
    tx, ty = left, max(0, top - 28)
    bbox = draw.textbbox((0,0), text)
    tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
    draw.rectangle([tx, ty, tx+tw+2*pad, ty+th+2*pad], fill=(0,0,0))
    draw.text((tx+pad, ty+pad), text, fill=(255,255,255))
    return img

def predict_and_save(img_path, topk=3, threshold=0.80, crop_box=True):
    preds = predict_image(img_path, topk=topk, threshold=threshold)
    best = preds[0]
    label, conf = best["class"], best["confidence"]
    status = "OK" if best["status"] == "ok" else "UNCERTAIN"

    img = Image.open(img_path).convert("RGB")
    if crop_box:
        img = annotate_center_crop(img, label, conf, crop_size=224)

    base = os.path.splitext(os.path.basename(img_path))[0]
    out_name = f"{base}__{label.replace(' ', '_')}__{status}__{conf:.3f}.jpg"
    out_path = os.path.join(OUT_DIR, out_name)
    img.save(out_path, quality=95)

    return out_path, preds


e:\semester7\comvis\deteksidaun\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Classes: ['Bercak Daun', 'Daun Sehat', 'Hawar Daun', 'Karat Daun']


In [7]:
IMG_PATH = r"E:\semester7\comvis\deteksidaun\testpredict\karat daun2.jpeg"
out_path, preds = predict_and_save(IMG_PATH, topk=3, threshold=0.80, crop_box=True)

print("Saved:", out_path)
for i, r in enumerate(preds, 1):
    print(f"Top-{i}: {r['class']} ({r['confidence']*100:.2f}%)")


Saved: E:\semester7\comvis\deteksidaun\outputs\karat daun2__Karat_Daun__OK__0.954.jpg
Top-1: Karat Daun (95.41%)
Top-2: Hawar Daun (1.86%)
Top-3: Daun Sehat (1.38%)
